# Known-stream search: example

How to make matched-filter maps of a known stream in the Euclid DR1 × DES Y6 footprint:

1. the stars and the survey footprint;
2. the galstreams streams crossing the footprint, at dm > 14;
3. for one stream: its frame, the window of the maps, its isochrones;
4. the map at one distance modulus;
5. the maps over the grid of distance moduli, with the video and the file of maps;
6. maps that follow the distance of the track;
7. the same from the stream name alone, and in one call for all the streams.

The maps are star counts in the match filter, in the stream frame (phi1, phi2). No background is subtracted. There is one set of maps per isochrone.

**Phase 2 stub.** Only the import cell runs: the other cells call functions that are not written yet (they raise `NotImplementedError`). Phase 3 makes this notebook run end to end on a validation stream (Section 7 of `docs/dev/known_streams_context.md`).

In [ ]:
from pathlib import Path

import healpy as hp
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from streameucliddes.analysis.known_stream_search import (
    known_stream_search_dm_along_track,
    known_stream_search_dm_scan,
    known_stream_search_single_dm,
    prepare_search,
)
from streameucliddes.catalogs.euclid_des_catalog import get_eucliddes_stars
from streameucliddes.catalogs.galstreams_catalog import (
    galstreams_phi12_window,
    load_galstreams,
    select_galstreams_in_footprint,
)
from streameucliddes.catalogs.survey_footprint import survey_footprint_healpix_mask
from streameucliddes.catalogs.isochrone_parameters import known_stream_isochrone_parameters
from streameucliddes.plotting import plot_style
from streameucliddes.plotting.plot_cmd import plot_match_filter
from streameucliddes.plotting.make_videos import make_gif_from_plots
from streameucliddes.plotting.plot_known_stream_search import (
    plot_galstreams_selection,
    plot_known_stream_search_single_dm,
)
from streameucliddes.stream_frames import radec_to_phi12
from streameucliddes.workflows.run_config import KNOWN_STREAM_SEARCH_CONFIG
from streameucliddes.workflows.run_known_stream_search import (
    run_all_known_stream_searches,
    run_known_stream_search,
    save_known_stream_search_maps,
)

## 1. Stars and footprint

The Euclid × DES stars. Their magnitudes are already dereddened: there is no dust correction below.

**Until the Euclid DR1 data are available**, `USE_FAKE_CATALOG = True` makes fake stars instead, uniform on the sphere over ra -20 to 100 and dec -70 to 0 (about 6,500 deg², like DES), with Euclid VIS and H, the bands of the default match filter, uniform in colour-magnitude space (VIS from 18 to 26, VIS - H from -1 to 1.5) and errors of 0.02 mag: a background without any stream, to run the pipeline. That area covers the DES streams of the validation (ATLAS, Phoenix, Turranburra; Shipp et al. 2018) and the windows of their maps.

The footprint comes from the star counts for now: a HEALPix pixel is in the footprint when it holds a star. A mask is True outside the footprint.

In [ ]:
USE_FAKE_CATALOG = True  # no Euclid DR1 data yet; False: the Euclid x DES catalog

if USE_FAKE_CATALOG:
    rng = np.random.default_rng(42)
    N_FAKE_STARS = 5_000_000  # about 10 per HEALPix pixel at nside 512: no holes in the footprint
    RA_RANGE, DEC_RANGE = (-20.0, 100.0), (-70.0, 0.0)  # deg
    ra = rng.uniform(*RA_RANGE, N_FAKE_STARS) % 360.0
    # Uniform on the sphere: uniform in sin(dec)
    dec = np.degrees(np.arcsin(rng.uniform(*np.sin(np.radians(DEC_RANGE)), N_FAKE_STARS)))
    # Euclid VIS and H uniform in colour-magnitude space (no stream), small errors (pass the default cuts)
    vis = rng.uniform(18.0, 26.0, N_FAKE_STARS)
    h = vis - rng.uniform(-1.0, 1.5, N_FAKE_STARS)
    errors = np.full(N_FAKE_STARS, 0.02)
    stars = pd.DataFrame({
        "ra": ra, "dec": dec,
        "euclid_q1_VIS_obs": vis, "euclid_q1_H_obs": h, "euclid_q1_VIS_err": errors, "euclid_q1_H_err": errors,
    })

In [ ]:
if not USE_FAKE_CATALOG:
    stars = get_eucliddes_stars()

NSIDE_FOOTPRINT = 512
footprint = survey_footprint_healpix_mask(stars["ra"], stars["dec"], NSIDE_FOOTPRINT, nest=False)
area = (~footprint).sum() * hp.nside2pixarea(NSIDE_FOOTPRINT, degrees=True)
print(f"{len(stars):,} stars; footprint: {area:,.0f} deg2")

## 2. Known streams in the footprint

The galstreams tracks whose part in the footprint, at dm > 14, is at least 1 deg long. The Ibata et al. 2024 tracks, without distances in galstreams, are left out (`add_ibata2024=True` to add them). Loading galstreams is slow: it is done once.

In [ ]:
mws = load_galstreams()
selection = select_galstreams_in_footprint(mws, footprint, nest=False)
selection

In [ ]:
fig, ax = plot_galstreams_selection(selection, mws, footprint, nest=False)

## 3. One stream: frame, window and isochrones

- The frame is the galstreams frame of the track: a gala `GreatCircleICRSFrame`.
- The window covers the track, with margins.
- The isochrones use the age and [Fe/H] of the stream when they are known (`stream_populations.csv`); otherwise the default isochrone, that of the DES search (Shipp et al. 2018): 13 Gyr, Z = 0.0002. Z is on the scale of the Marigo et al. 2017 isochrones used here. Each isochrone gives its own maps: below, the first one.

Here, ATLAS, a validation stream (Shipp et al. 2018), inside the fake footprint; any track of `selection` works.

In [ ]:
TRACK_NAME = "AAU-ATLAS-L21"
stream_name = selection.set_index("track_name").loc[TRACK_NAME, "stream_name"]

track = mws[TRACK_NAME]
stream_frame = track.stream_frame
window = galstreams_phi12_window(track)
track_phi12 = radec_to_phi12(track.track.ra.deg, track.track.dec.deg, stream_frame)
print(f"{TRACK_NAME} ({stream_name}): window phi1 {window[0]:.1f} to {window[1]:.1f}, phi2 {window[2]:.1f} to {window[3]:.1f} deg")

In [ ]:
isochrones = known_stream_isochrone_parameters(stream_name)  # reads stream_populations.csv
isochrone = isochrones[0]  # one set of maps per isochrone
isochrones

## 4. Map at one distance modulus

The match filter keeps the stars near the isochrone at this distance modulus. They are counted on the phi1/phi2 grid and smoothed. Bins outside the footprint are masked, and the counts are corrected at the footprint edges.

First, the search is prepared once, with `prepare_search`: the stars of the window passing the cuts, with their phi1/phi2; the footprint on the phi1/phi2 grid; the distance moduli of the track in the footprint (`dm_min` and `dm_max`, as in the selection); the isochrones of the stream. The maps below reuse it (`search=search`). The distance modulus of the map is the middle of `dm_min` and `dm_max`, the default of `known_stream_search_single_dm`.

In [ ]:
BIN_SIZE_DEG = 0.1
SMOOTHING_DEG = 0.3
MATCH_FILTER = {  # Euclid VIS vs VIS - H: the default match filter
    "name": "euclid_vis_h",
    "reference_band": ("euclid_q1", "VIS"),
    "color_bands": [("euclid_q1", "H")],
}

search = prepare_search(
    stars, stream_name=TRACK_NAME, mws=mws, match_filter=MATCH_FILTER,
    bin_size_deg=BIN_SIZE_DEG, footprint_mask=footprint, nest=False,
)
footprint_phi12 = search.footprint_mask
print(f"{len(search.stars):,} stars in the window; distance moduli of the track in the footprint: "
      f"{search.dm_min:.2f} to {search.dm_max:.2f}; isochrones {search.isochrones}")

phi1_centres, phi2_centres = search.phi1_centres, search.phi2_centres
half_bin = BIN_SIZE_DEG / 2
extent = (phi1_centres[0] - half_bin, phi1_centres[-1] + half_bin, phi2_centres[0] - half_bin, phi2_centres[-1] + half_bin)
fig, ax = plt.subplots(figsize=(8, 3))
ax.imshow(~footprint_phi12.T, origin="lower", extent=extent, aspect="auto", cmap="Greys", vmin=0, vmax=2)
ax.plot(*track_phi12, color="C1", label=TRACK_NAME)
ax.set(xlim=(extent[1], extent[0]), xlabel=r"$\phi_1$ [deg]", ylabel=r"$\phi_2$ [deg]", title="Footprint (grey) in the stream frame")
ax.legend(loc="upper right")
plt.show()

In [ ]:
result = known_stream_search_single_dm(isochrone=isochrone, search=search, smoothing_deg=SMOOTHING_DEG)
print(f"map at dm {result['distance_modulus']:.1f}: {result['n_selected']:,} stars in the match filter")
fig, ax = plot_known_stream_search_single_dm(result, track_phi12=track_phi12)

The colour-magnitude diagram of the stars of the window, with the match filter of this map: magnitude VIS against colour VIS - H, the convention of the match-filter polygons. `plot_match_filter` takes the stars prepared by `prepare_search` (`stars=search.stars`): the same columns and cuts as the map. With `catalog=` instead, it prepares the stars itself (`prepare_match_filter_catalog`; `stream_frame` and `window` restrict them to the window).

In [ ]:
fig, axes = plot_match_filter(result["distance_modulus"], isochrone, MATCH_FILTER, stars=search.stars,color_lim=(-1,1), mag_lim=(18,27),)

## 5. Scan over distance moduli: figures, video and file of maps

`known_stream_search_dm_scan` gives the maps one distance modulus at a time. Each one is drawn and saved; then the frames make the video, and the maps are saved together in one file. The phi1/phi2 maps are small: keeping them all is fine.

The outputs go to `figures/outputs/` (gitignored).

In [ ]:
isochrone_label = f"age{isochrone['age']:g}_z{isochrone['metallicity']:g}"
output_dir = Path("../figures/outputs/known_stream_search") / TRACK_NAME / isochrone_label
output_dir.mkdir(parents=True, exist_ok=True)
distance_moduli = np.arange(15.0, 22.0 + 0.05, 0.1)

results = []
for dm, res in known_stream_search_dm_scan(
    distance_moduli=distance_moduli, isochrone=isochrone, search=search, smoothing_deg=SMOOTHING_DEG,
):
    fig, ax = plot_known_stream_search_single_dm(res, track_phi12=track_phi12)
    fig.savefig(output_dir / f"{TRACK_NAME}_dm{dm:.2f}.png", dpi=110)
    plt.close(fig)
    results.append((dm, res))

frames = str(output_dir / f"{TRACK_NAME}_dm*.png")
make_gif_from_plots(output_dir / f"{TRACK_NAME}.gif", pattern=frames, duration=300)
save_known_stream_search_maps(
    results, output_dir / f"{TRACK_NAME}_maps.npz",
    metadata={"track_name": TRACK_NAME, "isochrone": isochrone},
)

The match filter over the same distance moduli, as a gif. The colour and magnitude ranges are fixed, so that the axes do not move from frame to frame.

In [ ]:
CMD_COLOR_LIM, CMD_MAG_LIM = (-1.0, 1.5), (17.0, 27.0)  # VIS - H, VIS
cmd_dir = output_dir / "cmd"
cmd_dir.mkdir(exist_ok=True)
for dm in distance_moduli:
    fig, axes = plot_match_filter(
        dm, isochrone, MATCH_FILTER, stars=search.stars, color_lim=CMD_COLOR_LIM, mag_lim=CMD_MAG_LIM,
    )
    fig.savefig(cmd_dir / f"{TRACK_NAME}_cmd_dm{dm:.2f}.png", dpi=110)
    plt.close(fig)
make_gif_from_plots(
    output_dir / f"{TRACK_NAME}_cmd.gif", pattern=str(cmd_dir / f"{TRACK_NAME}_cmd_dm*.png"), duration=300,
)

## 6. Following the distance of the track

A stream can have a distance gradient. Here the match filter follows the distance modulus of the track along phi1, plus an offset: one map per offset. Beyond the ends of the track, the distance modulus is extrapolated linearly from the track, to follow a possible extension of the stream.

In [ ]:
for dm_offset, res in known_stream_search_dm_along_track(
    isochrone=isochrone, search=search, dm_offsets=(-0.5, 0.0, 0.5), smoothing_deg=SMOOTHING_DEG,
):
    fig, ax = plot_known_stream_search_single_dm(
        res, track_phi12=track_phi12, title=f"{TRACK_NAME}, dm offset {dm_offset:+.1f}"
    )

## 7. From the stream name alone, and for all the streams

With only the stream name (a galstreams stream, or a track), the search takes what is not given from galstreams and `stream_populations.csv`: the default track, its frame and window, the distance moduli of the track ± 1 mag, and every isochrone of the stream. Any value given is kept. A HEALPix footprint is put on the grid of the window.

The workflows do steps 4 and 5 for each isochrone of the stream: the maps and the CMD with the match filter over the distance moduli, their gifs, and the file of maps, in `output_dir/<track>/age<age>_z<Z>/`. `run_known_stream_search` runs one stream (from its name, or a prepared search); `run_all_known_stream_searches` one track per stream crossing the footprint (its galstreams default track, else its longest one in the footprint), or the `track_names` given.

Their arguments, with the default values, are in `KNOWN_STREAM_SEARCH_CONFIG`: copy it and change what you need. (A YAML file of `configs/` can also change them, for the scripts: `RunConfig.from_yaml`.)

In [ ]:
counts = [
    {"dm": dm, "age": res["isochrone"]["age"], "z": res["isochrone"]["metallicity"], "n_selected": res["n_selected"]}
    for dm, res in known_stream_search_dm_scan(stars, stream_name="AAU-ATLAS", mws=mws, footprint_mask=footprint, nest=False)
]
print(f"track {res['track_name']}, window {tuple(round(x, 1) for x in res['window'])}")
pd.DataFrame(counts)

In [ ]:
OUTPUT_DIR = "../figures/outputs/known_stream_search"

summary = run_known_stream_search(search=search, output_dir=OUTPUT_DIR)  # the search prepared in section 4
for outputs in summary["isochrones"]:
    print(outputs["isochrone"], outputs["gif_path"], outputs["cmd_gif_path"], outputs["maps_path"], sep="\n  ")

In [ ]:
config = {**KNOWN_STREAM_SEARCH_CONFIG, "output_dir": OUTPUT_DIR}

# Without track_names: every stream crossing the footprint (long on the full catalog). Here, two tracks
table = run_all_known_stream_searches(
    stars, track_names=list(selection["track_name"][:2]), mws=mws, footprint_mask=footprint, nest=False, **config,
)
table[["track_name", "stream_name", "age", "metallicity", "gif_path", "maps_path"]]